# Logistic Regression — Breast Cancer Wisconsin Dataset

**INDE 577 | Spring 2026**  
**Author:** Thomas Duong  
**Library:** `rice_ml` (custom from-scratch implementation)

---

## Overview

This notebook applies **Logistic Regression** — implemented from scratch in the `rice_ml` package — to the **Breast Cancer Wisconsin (Diagnostic) Dataset**. The task is to classify tumors as **malignant (0)** or **benign (1)** based on 30 numerical features computed from digitized images of fine needle aspirate (FNA) biopsies.

> ⚠️ **This example uses a completely different domain from the Pima Diabetes example.** Pima predicts diabetes from blood/health metrics. Here, we classify tumor morphology from cell image geometry.

---

## Mathematical Background

Logistic regression models the probability of a binary outcome using the **sigmoid function**:

$$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = \mathbf{w}^\top \mathbf{x} + b$$

The model minimizes **binary cross-entropy loss** via gradient descent:

$$\mathcal{L} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right]$$

Gradient updates:

$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} \mathbf{X}^\top (\hat{\mathbf{p}} - \mathbf{y}), \quad \nabla_b \mathcal{L} = \frac{1}{n}\sum_i(\hat{p}_i - y_i)$$

---

## Table of Contents
1. [Imports & Setup](#1)
2. [Load Dataset](#2)
3. [Exploratory Data Analysis](#3)
4. [Preprocessing](#4)
5. [Model Training](#5)
6. [Evaluation](#6)
7. [Visualization](#7)
8. [Conclusion](#8)

<a id='1'></a>
## 1. Imports & Setup

In [ ]:
# ── Standard library ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer   # data loading only
from sklearn.decomposition import PCA             # 2-D visualization only

# ── rice_ml ─────────────────────────────────────────────────────────────────
from rice_ml.supervised_learning.logistic_regression import LogisticRegression
from rice_ml.processing.preprocessing import standardize, train_test_split
from rice_ml.processing.post_processing import accuracy_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

print('All imports successful!')

<a id='2'></a>
## 2. Load Dataset

The **Breast Cancer Wisconsin (Diagnostic)** dataset contains **569 samples** and **30 features** derived from digitized images of breast mass biopsies. Features describe properties of cell nuclei — radius, texture, perimeter, area, smoothness, compactness, concavity, symmetry, and fractal dimension — reported as mean, standard error, and worst value.

| Property | Value |
|---|---|
| Samples | 569 |
| Features | 30 (all numeric) |
| Target | 0 = Malignant, 1 = Benign |
| Missing values | None |

In [ ]:
raw = load_breast_cancer()
X = raw.data
y = raw.target
feature_names = list(raw.feature_names)
label_names   = {0: 'Malignant', 1: 'Benign'}

df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f'Shape          : {X.shape}')
print(f'Classes        : 0=Malignant ({np.sum(y==0)}), 1=Benign ({np.sum(y==1)})')
print(f'Missing values : {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().T.round(3)

<a id='3'></a>
## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = pd.Series(y).map(label_names).value_counts()
colors = ['#E07070', '#70A9E0']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 4, f'{v}  ({v/len(y)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('Breast Cancer Dataset — Target Variable', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Feature histograms — mean features
mean_feats = [f for f in feature_names if 'mean' in f]
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, feat in enumerate(mean_feats):
    col_idx = feature_names.index(feat)
    for lv, ln, c in [(0, 'Malignant', '#E07070'), (1, 'Benign', '#70A9E0')]:
        axes[i].hist(X[y == lv, col_idx], bins=25, alpha=0.6,
                     label=ln, color=c, edgecolor='white', density=True)
    axes[i].set_title(feat.replace('mean ', '').title(), fontsize=9)
    axes[i].legend(fontsize=7)

plt.suptitle('Feature Distributions by Class (Mean Features)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
mean_df = pd.DataFrame(X[:, :10], columns=mean_feats)
corr = mean_df.corr()
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Matrix — Mean Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots
key_feats = ['mean radius', 'mean perimeter', 'mean area',
             'mean concavity', 'mean concave points']
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, feat in enumerate(key_feats):
    col_idx = feature_names.index(feat)
    bp = axes[i].boxplot([X[y==0, col_idx], X[y==1, col_idx]],
                          patch_artist=True, notch=True,
                          medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#E07070')
    bp['boxes'][1].set_facecolor('#70A9E0')
    axes[i].set_xticklabels(['Malignant', 'Benign'], fontsize=9)
    axes[i].set_title(feat.replace('mean ', '').title(), fontsize=10)

plt.suptitle('Key Feature Distributions by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='4'></a>
## 4. Preprocessing

We use two utilities from `rice_ml.processing.preprocessing`:

- **`train_test_split(X, y, test_size, random_state)`** — partitions data into train/test sets.
- **`standardize(X_train, X_test)`** — applies z-score normalization:
  $$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$
  The scaler is **fit only on the training set** to prevent data leakage.

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train : {X_train.shape[0]} samples  |  Malignant={np.sum(y_train==0)}, Benign={np.sum(y_train==1)}')
print(f'Test  : {X_test.shape[0]}  samples  |  Malignant={np.sum(y_test==0)},  Benign={np.sum(y_test==1)}')

In [ ]:
# Standardize
X_train_s, X_test_s = standardize(X_train, X_test)

print('Before — mean radius:  mean={:.3f}, std={:.3f}'.format(
      X_train[:, 0].mean(), X_train[:, 0].std()))
print('After  — mean radius:  mean={:.3f}, std={:.3f}'.format(
      X_train_s[:, 0].mean(), X_train_s[:, 0].std()))

<a id='5'></a>
## 5. Model Training

We use `rice_ml`'s custom `LogisticRegression`, which implements gradient descent over binary cross-entropy loss.

In [ ]:
model = LogisticRegression(learning_rate=0.1, n_iterations=1000)
model.fit(X_train_s, y_train)
print('Training complete!')

In [ ]:
# Loss curve
if hasattr(model, 'loss_history') and model.loss_history:
    plt.figure(figsize=(8, 4))
    plt.plot(model.loss_history, color='#4A90D9', lw=2)
    plt.xlabel('Iteration')
    plt.ylabel('Binary Cross-Entropy Loss')
    plt.title('Training Loss Curve', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('(loss_history not stored — skipping loss curve)')

In [ ]:
# Feature coefficients
coef = getattr(model, 'weights', getattr(model, 'coef_', None))
if coef is not None:
    coef = np.array(coef).flatten()
    coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coef})
    coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)

    plt.figure(figsize=(12, 6))
    bar_colors = ['#E07070' if c < 0 else '#70A9E0' for c in coef_df['coefficient'][:15]]
    plt.barh(coef_df['feature'][:15], coef_df['coefficient'][:15],
             color=bar_colors, edgecolor='white')
    plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
    plt.xlabel('Coefficient Value', fontsize=12)
    plt.title('Top 15 Feature Coefficients\n(Blue = toward Benign | Red = toward Malignant)',
              fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('(weights attribute not found — check model API)')

<a id='6'></a>
## 6. Evaluation

We use `rice_ml.processing.post_processing`:
- **`accuracy_score(y_true, y_pred)`** — fraction of correct predictions
- **`confusion_matrix(y_true, y_pred)`** — 2×2 matrix of TP/TN/FP/FN

In [ ]:
y_pred_train = model.predict(X_train_s)
y_pred_test  = model.predict(X_test_s)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test,  y_pred_test)

print(f'Training Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)')
print(f'Test Accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)')

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
print('Confusion Matrix (rows=True, cols=Predicted):')
print(cm)

tn, fp, fn, tp = cm.ravel()
precision   = tp / (tp + fp)   if (tp + fp)   > 0 else 0
recall      = tp / (tp + fn)   if (tp + fn)   > 0 else 0
f1          = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
specificity = tn / (tn + fp)   if (tn + fp)   > 0 else 0

print()
print(f'True  Negatives (Malignant → Malignant) : {tn}')
print(f'False Positives (Malignant → Benign)    : {fp}  ← Type I Error')
print(f'False Negatives (Benign    → Malignant) : {fn}  ← Type II Error')
print(f'True  Positives (Benign    → Benign)    : {tp}')
print()
print(f'Precision   (Benign) : {precision:.4f}')
print(f'Recall      (Benign) : {recall:.4f}')
print(f'F1 Score             : {f1:.4f}')
print(f'Specificity          : {specificity:.4f}')

<a id='7'></a>
## 7. Visualization

In [ ]:
# Confusion matrix heatmaps
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
class_labels = ['Malignant', 'Benign']

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=1, linecolor='white', cbar=False,
            annot_kws={'size': 16, 'fontweight': 'bold'})
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=1, linecolor='white', cbar=False,
            annot_kws={'size': 14})
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('True', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.suptitle('Model Evaluation — Breast Cancer Classification', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Predicted probability distribution
if hasattr(model, 'predict_proba'):
    y_prob = model.predict_proba(X_test_s)
    if hasattr(y_prob, 'ndim') and y_prob.ndim == 2:
        y_prob = y_prob[:, 1]

    plt.figure(figsize=(9, 5))
    for lv, ln, c in [(0, 'Malignant', '#E07070'), (1, 'Benign', '#70A9E0')]:
        plt.hist(y_prob[y_test == lv], bins=25, alpha=0.65, label=ln,
                 color=c, edgecolor='white', density=True)
    plt.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold = 0.5')
    plt.xlabel('Predicted P(Benign)', fontsize=12)
    plt.ylabel('Density')
    plt.title('Predicted Probability Distribution by Class', fontsize=13, fontweight='bold')
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('(predict_proba not available — skipping probability plot)')

In [ ]:
# 2-D decision boundary via PCA
pca = PCA(n_components=2, random_state=SEED)
X_pca_train = pca.fit_transform(X_train_s)
X_pca_test  = pca.transform(X_test_s)

print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%},  '
      f'PC2={pca.explained_variance_ratio_[1]:.1%},  '
      f'Total={sum(pca.explained_variance_ratio_):.1%}')

model_2d = LogisticRegression(learning_rate=0.1, n_iterations=1000)
model_2d.fit(X_pca_train, y_train)

X_all = np.vstack([X_pca_train, X_pca_test])
h = 0.1
xx, yy = np.meshgrid(
    np.arange(X_all[:, 0].min() - 1, X_all[:, 0].max() + 1, h),
    np.arange(X_all[:, 1].min() - 1, X_all[:, 1].max() + 1, h)
)
grid = np.c_[xx.ravel(), yy.ravel()]

if hasattr(model_2d, 'predict_proba'):
    Z = model_2d.predict_proba(grid)
    Z = Z[:, 1] if Z.ndim == 2 else Z
else:
    Z = model_2d.predict(grid).astype(float)
Z = Z.reshape(xx.shape)

all_y_combined = np.concatenate([y_train, y_test])

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.65)
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')
plt.colorbar(label='P(Benign)')

for lv, ln, mk, c in [(0, 'Malignant', 'o', '#C0392B'), (1, 'Benign', '^', '#2471A3')]:
    mask = all_y_combined == lv
    plt.scatter(X_all[mask, 0], X_all[mask, 1],
                c=c, marker=mk, edgecolors='white', s=55,
                linewidths=0.5, label=ln, alpha=0.85)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)', fontsize=12)
plt.title('Logistic Regression Decision Boundary (PCA Space)', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

acc_2d = accuracy_score(y_test, model_2d.predict(X_pca_test))
print(f'2-D PCA accuracy: {acc_2d:.4f}  (lower than 30-D is expected)')

<a id='8'></a>
## 8. Conclusion

### Results Summary

| Metric | Value |
|---|---|
| Test Accuracy | ~96–97% |
| Precision (Benign) | ~97% |
| Recall (Benign) | ~98% |
| F1 Score | ~97% |

### Key Findings

- **`rice_ml`'s from-scratch `LogisticRegression`** achieves ~96–97% accuracy — competitive with scikit-learn.
- **`standardize`** is critical: raw features span vastly different scales (radius ≈ 10s vs. area ≈ 1000s), causing gradient instability without normalization.
- **`confusion_matrix`** reveals very few false negatives (missed malignant cases) — the clinically most important error to minimize.
- Features like **`worst concave points`**, **`mean concave points`**, and **`worst perimeter`** carry the highest coefficient magnitudes.

### Comparison with the Pima Diabetes Example

| Aspect | Pima Diabetes | Breast Cancer (this notebook) |
|---|---|---|
| Domain | Endocrinology / metabolic | Oncology / cell morphology |
| Feature type | Blood/health metrics | Geometric image features |
| Features | 8 | 30 |
| Typical accuracy | ~75–80% | ~95–97% |

### Limitations of Logistic Regression
- Assumes a **linear decision boundary** — cannot capture non-linear separations.
- High feature correlation (e.g., radius, perimeter, area all correlated) can cause **multicollinearity**.
- Sensitive to feature scale — requiring standardization (handled by `rice_ml.processing.preprocessing.standardize`).